# S8b 인용 의미 판정(JEV) 파일럿

원문에 그대로 있지 않은 인용을 TypeSafe Jev(`typesafe/jev-1.13`)로 판정할 수 있는지, 같은 모델로 인용이 사실을 뒷받침하는지도 판정할 수 있는지 보는 근거 자료다(`docs/slices/08b-quote-semantic.md`). 이 노트북을 보고 기준값(`min_prob`, `top_k`, `window_sentences`)과 뒷받침 판정 채택 여부를 정한다.

- Jev 호출은 `data/multidoc/quote_semantic/calls.jsonl`에 캐시한다. 다시 실행하면 캐시에 없는 요청만 호출한다
- 음성 2(뜻을 바꾼 변형)는 메인이 손으로 쓴 `data/multidoc/quote_semantic/neg2.json`을 읽는다
- 회의록 원문은 짧은 예시로만 출력한다

In [1]:
import hashlib
import json
import os
import random
import re
import time
import urllib.request
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from rapidfuzz import fuzz, process

from rag.index import load_cfg
from rag.multidoc.check import norm

In [2]:
ROOT = Path.cwd().parent
cfg = load_cfg()
CHECK = ROOT / cfg.paths.multidoc_check
DOCS = ROOT / cfg.paths.multidoc_docs
OUT = ROOT / "data/multidoc/quote_semantic"
MODEL = "typesafe/jev-1.13"
URL = "https://openrouter.ai/api/alpha/decisions"
TOP_K = 3  # 후보 대목 수
WINDOW_SENTENCES = 3  # 후보 대목 하나에 묶는 연속 문장 수
SEED = 20261007
N_CONTROL = 10  # 원문에 그대로 있는 인용 대조 묶음
N_SUPPORT = 30  # 뒷받침 양성·음성 묶음 크기
N_REPEAT = 5  # 결정성 확인용 반복 호출
SAME = {
    "type": "noul",
    "instructions": "후보 대목 가운데 인용과 같은 의미의 내용이 있는가?",
    "criteria": {
        "true": (
            "후보 대목 하나 이상이 인용과 같은 사실·주장·질문을 말한다. "
            "접속어, 존칭, 어순, 낱말 표현이 달라도 된다."
        ),
        "false": (
            "어느 후보 대목도 인용과 같은 의미가 아니다. "
            "숫자·날짜·부정·주체·대상이 다르거나, 인용의 내용이 대목에 없다."
        ),
    },
}
SUPPORT = {
    "type": "noul",
    "instructions": "인용이 사실을 뒷받침하는가?",
    "criteria": {
        "true": "인용의 내용으로 사실의 핵심(누가, 무엇을, 어떤 수치나 입장)이 확인된다.",
        "false": ("인용이 사실과 다른 내용을 말하거나, 인용으로 사실의 핵심을 확인할 수 없다."),
    },
}
load_dotenv(ROOT / ".env")

True

In [3]:
_ctx = {}


def context(doc_id):
    if doc_id not in _ctx:
        _ctx[doc_id] = norm((DOCS / f"{doc_id}.txt").read_text(encoding="utf-8"))
    return _ctx[doc_id]


def candidates(quote, doc_id):
    # 문장 끝 부호로 나눈 문장을 WINDOW_SENTENCES개씩 묶어 한 문장씩 민 대목 중
    # partial_ratio 상위 TOP_K개
    sents = re.split(r"(?<=[.?!…])\s+", context(doc_id))
    windows = [
        " ".join(sents[i : i + WINDOW_SENTENCES])
        for i in range(max(1, len(sents) - WINDOW_SENTENCES + 1))
    ]
    return [
        w
        for w, _, _ in process.extract(norm(quote), windows, scorer=fuzz.partial_ratio, limit=TOP_K)
    ]


def load_cache():
    path = OUT / "calls.jsonl"
    if not path.exists():
        return {}
    return {r["key"]: r for r in map(json.loads, path.read_text(encoding="utf-8").splitlines())}


cache = load_cache()


def jev(state, question, tag=""):
    body = {"model": MODEL, "state": state, "questions": {"q": question}}
    raw = json.dumps(body, ensure_ascii=False, sort_keys=True) + tag
    key = hashlib.sha256(raw.encode()).hexdigest()
    if key not in cache:
        req = urllib.request.Request(
            URL,
            data=json.dumps(body).encode(),
            headers={
                "Authorization": "Bearer " + os.environ["OPENROUTER_API"],
                "Content-Type": "application/json",
            },
        )
        t = time.time()
        with urllib.request.urlopen(req, timeout=60) as r:
            resp = json.loads(r.read().decode())
        rec = {"key": key, "tag": tag, "latency": round(time.time() - t, 3), "response": resp}
        with (OUT / "calls.jsonl").open("a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        cache[key] = rec
    rec = cache[key]
    u = rec["response"]["usage"]
    return {
        "p": rec["response"]["answers"]["q"]["noul"],
        "latency": rec["latency"],
        "input_tokens": u["input_tokens"],
        "cost": u["cost"],
    }

## 1. 묶음 만들기

In [4]:
recs = [json.loads(f.read_text(encoding="utf-8")) for f in sorted(CHECK.glob("*.json"))]
recs = [r for r in recs if r.get("elements")]
quotes = []  # (pool_id, 쪽, 요소 번호, 사실, doc_id, 인용)
for r in recs:
    for e in r["gen"]["evidence"]:
        quotes.append((r["pool_id"], "gen", None, None, e["doc_id"], e["quote"]))
    for i, el in enumerate(r["elements"]):
        for s in el["support"]:
            if s["doc_id"]:
                quotes.append((r["pool_id"], "check", i, el["fact"], s["doc_id"], s["quote"]))
quotes = pd.DataFrame(quotes, columns=["pool_id", "side", "el", "fact", "doc_id", "quote"])
quotes["exact"] = [
    norm(q) in context(d) for q, d in zip(quotes["quote"], quotes["doc_id"], strict=True)
]
print(
    len(quotes),
    "인용, 원문에 그대로 있음",
    quotes["exact"].sum(),
    "| 없음",
    (~quotes["exact"]).sum(),
)
quotes.groupby(["side", "exact"]).size()

516 인용, 원문에 그대로 있음 502 | 없음 14


side   exact
check  False     14
       True     399
gen    True     103
dtype: int64

In [5]:
rng = random.Random(SEED)
pos = quotes[~quotes["exact"]].assign(group="양성")
control = quotes[quotes["exact"]].sample(N_CONTROL, random_state=SEED).assign(group="대조")
print(len(pos), len(control))

14 10


In [6]:
# 원문에 그대로 있는 검사 인용을 같은 후보 집합의 다른 문서에 대 본 쌍 중 partial_ratio 0.6 이상
neg1 = []
for r in recs:
    for el in r["elements"]:
        for s in el["support"]:
            q = norm(s["quote"])
            if not (s["doc_id"] and q in context(s["doc_id"])):
                continue
            for d in r["pool_doc_ids"]:
                if (
                    d != s["doc_id"]
                    and q not in context(d)
                    and fuzz.partial_ratio(q, context(d)) >= 60
                ):
                    neg1.append((r["pool_id"], "check", None, None, d, s["quote"]))
neg1 = pd.DataFrame(neg1, columns=["pool_id", "side", "el", "fact", "doc_id", "quote"]).assign(
    group="음성 1"
)
print(len(neg1), "쌍")

37 쌍


In [7]:
neg2 = pd.DataFrame(
    [
        {
            "pool_id": x["pool_id"],
            "side": "check",
            "el": x["el"],
            "fact": None,
            "doc_id": x["doc_id"],
            "quote": x["variant"],
            "kind": x["kind"],
        }
        for x in json.loads((OUT / "neg2.json").read_text(encoding="utf-8"))
    ]
).assign(group="음성 2")
neg2["kind"].value_counts()

kind
negation    11
subject     10
number       9
Name: count, dtype: int64

In [8]:
sup = quotes[(quotes["side"] == "check") & quotes["exact"]]
sup_pos = sup.sample(N_SUPPORT, random_state=SEED).assign(group="뒷받침 양성")
# 같은 후보 집합의 다른 요소에서 온 인용. 그 인용의 문서가 원래 요소의 근거 문서가 아닐 때만
mis = []
for r in recs:
    els = r["elements"]
    for i, el in enumerate(els):
        docs_i = {s["doc_id"] for s in el["support"]}
        for j, other in enumerate(els):
            if j != i:
                mis += [
                    (r["pool_id"], "check", i, el["fact"], s["doc_id"], s["quote"])
                    for s in other["support"]
                    if s["doc_id"]
                    and s["doc_id"] not in docs_i
                    and norm(s["quote"]) in context(s["doc_id"])
                ]
mis = pd.DataFrame(mis, columns=["pool_id", "side", "el", "fact", "doc_id", "quote"])
sup_neg = mis.sample(N_SUPPORT, random_state=SEED).assign(group="뒷받침 음성")
c5 = next(r for r in recs if r["pool_id"] == "conf-0005")
d2 = c5["pool_doc_ids"][1]  # G3 검수에서 질의와 무관하다고 본 D2 근거
g3 = pd.DataFrame(
    [
        ("conf-0005", "check", i, el["fact"], s["doc_id"], s["quote"])
        for i, el in enumerate(c5["elements"])
        for s in el["support"]
        if s["doc_id"] == d2
    ],
    columns=sup_neg.columns[:6],
).assign(group="뒷받침 음성(G3)")
print(len(sup_pos), len(sup_neg), len(g3))

30 30 1


In [9]:
pairs = pd.concat([pos, control, neg1, neg2, sup_pos, sup_neg, g3], ignore_index=True)
pairs["task"] = pairs["group"].map(lambda g: "support" if g.startswith("뒷받침") else "same")
pairs.drop(columns=["exact"]).to_json(
    OUT / "pairs.json", orient="records", force_ascii=False, indent=1
)
pairs.groupby(["task", "group"]).size()

task     group     
same     대조            10
         양성            14
         음성 1          37
         음성 2          30
support  뒷받침 양성        30
         뒷받침 음성        30
         뒷받침 음성(G3)     1
dtype: int64

## 2. 호출

In [10]:
rows = []
for x in pairs.itertuples():
    if x.task == "same":
        out = jev({"인용": norm(x.quote), "후보 대목": candidates(x.quote, x.doc_id)}, SAME)
    else:
        out = jev({"사실": x.fact, "인용": norm(x.quote)}, SUPPORT)
    rows.append({**x._asdict(), **out})
for x in pos.head(N_REPEAT).itertuples():
    out = jev(
        {"인용": norm(x.quote), "후보 대목": candidates(x.quote, x.doc_id)}, SAME, tag="repeat"
    )
    rows.append({**x._asdict(), "group": "반복", "task": "same", **out})
res = pd.DataFrame(rows)
print(len(res), "건, 캐시", len(cache))

157 건, 캐시 157


## 3. 의미 동일성

In [11]:
same = res[(res["task"] == "same") & (res["group"] != "반복")]
same.groupby("group")["p"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(3)

,count,min,25%,50%,75%,max
group,,,,,,
대조,10.0,0.94,0.970,0.970,0.980,0.98
양성,14.0,0.93,0.962,0.970,0.978,0.98
음성 1,37.0,0.06,0.100,0.230,0.450,0.96
음성 2,30.0,0.05,0.295,0.615,0.830,0.96


In [12]:
ok = same["group"].isin(["양성", "대조"])
thr = pd.DataFrame(
    [
        {
            "min_prob": t,
            "양성·대조 인정": int((same[ok]["p"] >= t).sum()),
            "음성 1 인정": int((same[same["group"] == "음성 1"]["p"] >= t).sum()),
            "음성 2 인정": int((same[same["group"] == "음성 2"]["p"] >= t).sum()),
        }
        for t in [0.3, 0.5, 0.7, 0.8, 0.9]
    ]
)
print(
    "양성·대조",
    int(ok.sum()),
    "| 음성 1",
    int((same["group"] == "음성 1").sum()),
    "| 음성 2",
    int((same["group"] == "음성 2").sum()),
)
thr

양성·대조 24 | 음성 1 37 | 음성 2 30


,min_prob,양성·대조 인정,음성 1 인정,음성 2 인정
0,0.3,24,15,22
1,0.5,24,9,19
2,0.7,24,3,12
3,0.8,24,3,10
4,0.9,24,2,4


In [13]:
neg2_res = same[same["group"] == "음성 2"]
neg2_res.groupby("kind")["p"].describe()[["count", "min", "50%", "max"]].round(3)

,count,min,50%,max
kind,,,,
negation,11.0,0.05,0.50,0.96
number,9.0,0.07,0.31,0.87
subject,10.0,0.23,0.74,0.86


In [14]:
# 같은 변형을 문자 유사도(PR #31 기준 partial_ratio 0.8)로 판정하면
r = [
    fuzz.partial_ratio(norm(q), context(d)) / 100
    for q, d in zip(neg2_res["quote"], neg2_res["doc_id"], strict=True)
]
print(
    "음성 2 문자 유사도 0.8 이상",
    sum(x >= 0.8 for x in r),
    "/",
    len(r),
    "| Jev 0.9 이상",
    int((neg2_res["p"] >= 0.9).sum()),
)

음성 2 문자 유사도 0.8 이상 28 / 30 | Jev 0.9 이상 4


In [15]:
# 기준값을 정할 때 걸리는 쌍: 낮은 양성·대조와 높은 음성
show = pd.concat([same[ok].nsmallest(5, "p"), same[~ok].nlargest(8, "p")])
show.assign(quote=show["quote"].map(lambda q: norm(q)[:60]))[
    ["group", "kind", "pool_id", "p", "quote"]
]

,group,kind,pool_id,p,quote
2,양성,NaN,conf-0013,0.93,"7월 14일 날 1차 발표하고, 7월 28일 날, 14일 만에 또다시 2차 권고안 ..."
15,대조,NaN,questioner-0006,0.94,"경찰이 하는 일은 없습니까, 그러면?"
20,대조,NaN,questioner-0005,0.94,"기재부장관님, 산하기관 지원 예산은 이미 다 내년도 삭제가 됐습니다. 그런데 그게 ..."
11,양성,NaN,law-0009,0.95,그러면 계도 기간을 두는 것은 어때요? 예를 들어서 한 2년이나 이렇게 계도 기간을...
9,양성,NaN,law-0007,0.96,기획재정부장관이 관리하는 기금뿐만 아니라 국가가 직접 관리하지 않는 기금까지 다 포...
50,음성 1,NaN,conf-0019,0.96,"그런데 스튜디오를 다시 만들 필요가 있어요, 별도로?"
71,음성 2,negation,conf-0014,0.96,그러니까 그 부분에 대해서는 새로 취임하셨으니까 중소기업에 힘이 되지 않는 TV홈쇼...
73,음성 2,negation,conf-0010,0.96,그리고 살충제 계란 사태가 사실은 심각한 사건이기는 하지만 또 훨씬 더 크게 증폭된...
78,음성 2,negation,questioner-0006,0.96,그때 류삼영 총경의 징계에 대한 논의는 없었습니다.
49,음성 1,NaN,conf-0018,0.95,"권순일, 박영수, 곽상도, 김수남, 최재경, 그리고 홍 모 씨입니다."


메인이 원문을 보고 확인한 라벨(2026-10-07):

- 음성 1에서 0.84 이상인 3쌍은 실제로 같은 의미다. conf-0019는 다른 문서에도 "별도로 스튜디오를 또 구축할 필요가 있나요?"가 있고, conf-0018은 같은 인명 나열이 다른 문서에도 있으며, questioner-0000은 "조금 더 많이 하지 왜 하나 해요?"가 있다. 다른 문서라는 이유로 음성으로 묶었지만 Jev 판정이 맞다. 이 3쌍을 빼면 음성 1의 최대는 0.66이다
- 음성 2에서 0.9 이상으로 인정된 4개는 모두 부정 변형이다("볼 수 없습니다", "논의는 없었습니다", "옳지 않습니다", "힘이 되지 않는 … 되게 하실 수는 없나요?"). 마지막은 변형 문장 자체가 어색해 뜻이 분명히 뒤집혔다고 보기 어렵다. 숫자·주체 변형은 0.9 이상이 없다

## 4. 결정성

In [16]:
rep = res[res["group"] == "반복"][["pool_id", "quote", "p"]].merge(
    same[same["group"] == "양성"][["pool_id", "quote", "p"]],
    on=["pool_id", "quote"],
    suffixes=("_반복", "_처음"),
)
rep.assign(diff=(rep["p_반복"] - rep["p_처음"]).abs())[["pool_id", "p_처음", "p_반복", "diff"]]

,pool_id,p_처음,p_반복,diff
0,conf-0005,0.97,0.97,0.00
1,conf-0013,0.97,0.96,0.01
2,conf-0013,0.93,0.94,0.01
3,conf-0014,0.97,0.96,0.01
4,conf-0016,0.97,0.97,0.00


## 5. 사실 뒷받침

In [17]:
supp = res[res["task"] == "support"]
supp.groupby("group")["p"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(3)

,count,min,25%,50%,75%,max
group,,,,,,
뒷받침 양성,30.0,0.16,0.582,0.695,0.818,0.90
뒷받침 음성,30.0,0.02,0.060,0.095,0.200,0.52
뒷받침 음성(G3),1.0,0.74,0.740,0.740,0.740,0.74


In [18]:
pd.DataFrame(
    [
        {
            "min_prob": t,
            "양성 인정": int((supp[supp["group"] == "뒷받침 양성"]["p"] >= t).sum()),
            "음성 인정": int((supp[supp["group"] != "뒷받침 양성"]["p"] >= t).sum()),
        }
        for t in [0.3, 0.5, 0.7, 0.8, 0.9]
    ]
)

,min_prob,양성 인정,음성 인정
0,0.3,27,7
1,0.5,24,2
2,0.7,15,1
3,0.8,10,0
4,0.9,1,0


In [19]:
# 라벨 확인 대상: 확률이 낮은 양성과 높은 음성. 사실과 인용은 앞부분만
chk = pd.concat(
    [
        supp[supp["group"] == "뒷받침 양성"].nsmallest(6, "p"),
        supp[supp["group"] != "뒷받침 양성"].nlargest(6, "p"),
    ]
)
chk.assign(fact=chk["fact"].str[:50], quote=chk["quote"].map(lambda q: norm(q)[:50]))[
    ["group", "pool_id", "el", "p", "fact", "quote"]
]

,group,pool_id,el,p,fact,quote
120,뒷받침 양성,law-0008,4.0,0.16,"약 4만 6천 개 규모의 대상 사업자를 세무서 등 국세청이 직접 검증할 수 있는지,...",그러니까 세무조사 하는 강도로 하는 것은 아니잖아요. 이 세무검증이라고 하는 것은…...
105,뒷받침 양성,law-0005,2.0,0.17,"회원에게 지급되는 보상금이나 생활보조금 등 국가 지원 여부도 고려요소로 삼되, 그것...","지금 위원님께서 말씀해 주신 사항하고 모두에 위원장님, 김기식 위원님이 말씀해 주신..."
91,뒷받침 양성,conf-0018,0.0,0.25,박수영 위원은 FIU가 2021년 4월 1일 화천대유의 이상한 자금 흐름을 확인해 ...,FIU가 수상한 자금 흐름을 확인하고 경찰에 통보했다고 국가수사본부는 우리한테 얘기...
101,뒷받침 양성,conf-0014,4.0,0.33,"중소업체들이 TV홈쇼핑에 들어가기 위해 마케팅 비용, 즉 협찬 비용을 부담한다는 점...",그러니까 그 부분에 대해서는 새로 취임하셨으니까 중소기업에 힘이 되는 TV홈쇼핑 되...
114,뒷받침 양성,questioner-0003,1.0,0.33,한일 관계 악화로 어려움을 겪었던 재일동포 상인들의 상황이 국교정상화 이후 90%까...,그러니까 한 90% 회복됐다는 것도 동의하신다는 이런 얘기지요?
99,뒷받침 양성,law-0001,0.0,0.47,"2007년 11월 19일에는 장해연금과 관련해, 부상한 공무원에게 일시보상금을 지급...",부상의 경우에 일시보상금이 없지 않습니까?
151,뒷받침 음성(G3),conf-0005,0.0,0.74,무급 전임자는 노동조합에서 보수를 받고 임용권자로부터 급여를 받지 않는 사람으로 보...,지금 무급 전임자의 경우에는 무급휴직이라는 제도가 있었기 때문에 전임 허가를 받으면...
136,뒷받침 음성,conf-0014,3.0,0.52,"TV홈쇼핑 판매수수료율은 약 30% 정도라고 답변되었고, T커머스는 평균 약 25%...","최근에 홈쇼핑에서 판매하던 여행업체, 107억 원 여행상품을 팔았다가 홈쇼핑은 19..."
127,뒷받침 음성,conf-0013,0.0,0.47,"경찰개혁위원회는 경찰청장과 참모들이 자체적으로 추진했다고 설명했지만, 법적 근거가 ...",추천은 두 분을 받았고 나머지는 자체적으로 했습니다.
131,뒷받침 음성,law-0009,1.0,0.46,2017년에는 사진 부착 금지를 즉시 적용하기보다 약 2년의 계도 기간을 둔 뒤 과...,"예, 기본적으로 이 부분을 법으로 금지하는 것은 아직은 좀 시기상조가 아닐까 그런 ..."


메인이 원문을 보고 확인한 라벨(2026-10-07). 확률이 가장 낮은 뒷받침 양성 6쌍:

- 뒷받침하지 않음 2: law-0008(사실은 4만 6천 개 사업자·인력 쟁점, 인용은 세무검증의 강도), law-0005(사실은 보상금이 고려요소라는 기준, 인용은 "세 가지를 다 고려해야")
- 일부만 뒷받침 3: conf-0018 el0, conf-0014 el4, law-0001 el0. 사실이 질의와 답변처럼 여러 내용을 합친 문장이고, 인용은 그중 한 부분만 담는다
- Jev 오판 1: questioner-0003(인용 "90% 회복됐다는 것도 동의하신다는 이런 얘기지요?"가 사실 "90% 회복 주장에 동의하는지 물었다"를 그대로 뒷받침)
- 뒷받침 음성(G3) conf-0005 D2는 0.74로 높게 나왔다. 검수에서 질의와 무관하다고 본 근거다

## 6. 비용과 지연

In [20]:
calls = pd.DataFrame([{"latency": c["latency"], **c["response"]["usage"]} for c in cache.values()])
print(
    f"{len(calls)}호출 | 비용 합 ${calls['cost'].sum():.5f} | "
    f"입력 토큰 평균 {calls['input_tokens'].mean():.0f}, 최대 {calls['input_tokens'].max()} | "
    f"지연 중앙값 {calls['latency'].median():.2f}s, 최대 {calls['latency'].max():.2f}s"
)

157호출 | 비용 합 $0.00453 | 입력 토큰 평균 687, 최대 1279 | 지연 중앙값 0.43s, 최대 1.12s
